# MALMAS Deep Dive: From Prompts to Features

> ⚠️ **Most cells require an LLM API key.**
>
> This notebook traces the **MALMAS method** step-by-step: router selection, agent prompt assembly,
> LLM feature spec generation, code generation, sandbox execution, and evaluation.
> For other methods (CAAFE, LLM-FE, OpenFE, Malmus), see [05_methods_deep_dive.ipynb](./05_methods_deep_dive.ipynb).

## 1. Pipeline Overview

The MALMAS pipeline runs N rounds of feature engineering:

```
X_train, y_train
    │
    ▼
RouterAgent.select_agents() ──▶ [agent_names]        ← Section 4
    │
    ▼
For each agent (parallel):
    system_prompt + user_prompt ──▶ LLM ──▶ JSON      ← Section 5 (Feature Spec Generation)
    parse ──▶ list[FeatureSpec]
    │
    ▼
CodeGenerator.generate_code(specs, schema)             ← Section 6 (Code Generation)
    AST validation + ruff lint (retry once)
    │
    ▼
SandboxedExecutor.execute(code, df)                    ← Section 7 (Sandbox Execution)
    process-isolated, restricted imports
    │
    ▼
CVEvaluator.evaluate_feature() ──▶ gain per feature   ← Section 8 (Evaluation)
    select top-k with positive gain
    │
    ▼
AgentMemory.record_procedure/feedback()                ← Section 10 (Memory)
    RouterAgent.update_performance()
    X_train_enhanced for next round
```

## 2. Setup

The MALMAS method is Feature Forge's core multi-agent pipeline. All components live under `feature_forge.methods.malmas/`.

In [1]:
import json
import os
import pathlib
import sys
import time
import warnings

warnings.filterwarnings('ignore')
os.environ.setdefault('FF_LOG_LEVEL', 'warning')
sys.path.insert(0, str(pathlib.Path('.').resolve()))

import matplotlib.pyplot as plt
import nest_asyncio; nest_asyncio.apply()

from _utils import get_llm_client, make_sample_data
from feature_forge.config import RouterConfig, Settings
from feature_forge.types import FeatureSpec

X_train, X_test, y_train, y_test = make_sample_data(n_samples=250, n_features=8, n_informative=5)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

llm = get_llm_client()
if llm is None: print('No LLM key. LLM cells will be skipped.')

settings = Settings(n_rounds=3, metric='auc')
print(f'Default config: {settings.n_rounds} rounds, metric={settings.metric}')
print(f'Router: {settings.router.strategy}, CV folds: {settings.evaluation.cv_folds}')
print(f'LLM: {settings.llm.model}, thinking={settings.llm.thinking_enabled}')

Train: (175, 8), Test: (75, 8)
Default config: 3 rounds, metric=auc
Router: hybrid, CV folds: 5
LLM: deepseek-chat, thinking=False


## 3. Built-in Agents

MALMAS has 6 specialized agents, each generating features from a different angle:

| Agent | Focus | Example Features |
|-------|-------|------------------|
| `unary` | Single-column transforms | `log(age)`, `fare²` |
| `cross_compositional` | Multi-column interactions | `age × fare`, `ratio(a,b)` |
| `aggregation` | GroupBy aggregations | `mean(fare) by class` |
| `temporal` | Time-based features | `day_of_week`, `days_since` |
| `local_transform` | Ranking/quantile transforms | `rank(age)`, `quantile_bin` |
| `local_pattern` | Distribution patterns | `z_score`, `is_outlier` |

Each agent is a thin subclass of `BaseFeatureAgent` — it sets `prompt_key` and `agent_name`.
All LLM interaction logic lives in the base class (`agents/base.py`).

In [2]:
from feature_forge.methods.malmas.agents import AgentRegistry, BaseFeatureAgent
from feature_forge.methods.malmas.agents.router import RouterAgent

agents = AgentRegistry.get_builtin_agents()
print(f'{len(agents)} built-in agents:\n')
for name, cls in agents.items():
    caps = RouterAgent.AGENT_CAPABILITIES.get(name, {})
    desc = caps.get('description', 'N/A')
    print(f'  {name}: {cls.__name__} — {desc}')

class PolynomialAgent(BaseFeatureAgent):
    agent_name = 'polynomial'
    prompt_key = 'unary'

    @property
    def system_prompt(self):
        return 'Generate polynomial interaction features from numeric columns.'

print('\nCustom agents: subclass BaseFeatureAgent, set agent_name + prompt_key, add YAML prompt.')
print('Register via entry point: feature_forge.methods.malmas.agents')

6 built-in agents:

  unary: UnaryFeatureAgent — Generates features from single columns
  cross_compositional: CrossCompositionalAgent — Generates cross features between 2+ columns
  aggregation: AggregationConstructAgent — Generates aggregation-based features
  temporal: TemporalFeatureAgent — Generates time-based features
  local_transform: LocalTransformAgent — Generates local transformation features
  local_pattern: LocalPatternAgent — Generates features based on distributional patterns

Custom agents: subclass BaseFeatureAgent, set agent_name + prompt_key, add YAML prompt.
Register via entry point: feature_forge.methods.malmas.agents


## 4. Router Strategies

The router decides which agents run each round. 4 strategies:

| Strategy | Logic | Config |
|----------|-------|--------|
| `data_driven` | Matches column types to agent capabilities | `RouterConfig(strategy='data_driven')` |
| `performance_driven` | Keeps agents with positive average gain | Rolling window of last 10 gains |
| `hybrid` *(default)* | Union of data_driven + performance_driven | Best of both |
| `llm` | LLM chooses agents based on dataset context | Falls back to hybrid on error |

**Warmup**: Round 0 always uses all agents. Strategy kicks in from round 1+.

In [3]:
for strategy in ['data_driven', 'hybrid']:
    r = RouterAgent(Settings(router=RouterConfig(strategy=strategy)))
    selected = await r.select_agents(0, df=X_train, task_description='classification')
    print(f'{strategy}: {selected}')

r_perf = RouterAgent(Settings(router=RouterConfig(strategy='performance_driven')))
r_perf.update_performance('unary', 0.05)
r_perf.update_performance('cross_compositional', 0.03)
r_perf.update_performance('aggregation', -0.01)
selected = await r_perf.select_agents(1, df=X_train, task_description='classification')
print(f'performance_driven: {selected}')

if llm is None:
    print('Skipping LLM router \u2014 no API key.')
else:
    try:
        r_llm = RouterAgent(Settings(router=RouterConfig(strategy='llm')), llm_client=llm)
        selected = await r_llm.select_agents(0, df=X_train, task_description='classification')
        print(f'llm: {selected}')
    except Exception as e:
        print(f'LLM router error: {type(e).__name__}: {e}')

data_driven: ['unary', 'cross_compositional', 'aggregation', 'temporal', 'local_transform', 'local_pattern']
hybrid: ['unary', 'cross_compositional', 'aggregation', 'temporal', 'local_transform', 'local_pattern']
performance_driven: ['unary', 'cross_compositional']
llm: ['unary', 'cross_compositional', 'aggregation', 'temporal', 'local_transform', 'local_pattern']


## 5. Feature Spec Generation — The First LLM Stage

This is where the LLM generates feature specifications. For each agent:

1. **System prompt** loaded from `config/prompts/{agent_name}.yaml`
2. **User prompt** assembled from: task, column descriptions (JSON), memory context, positive/negative features
3. **JSON schema** injected into system message: `{"features": [{"base_columns": [...], "derived_features": [...]}]}`
4. **LLM response** parsed into `list[FeatureSpec]`

Let's trace this step-by-step.

In [4]:
from feature_forge.methods.malmas.agents.base import BaseFeatureAgent
from feature_forge.prompts import get_registry

registry = get_registry()
unary_prompt = registry.get('unary')

print('=== SYSTEM PROMPT (from config/prompts/unary.yaml) ===\n')
print(unary_prompt.system[:600] + '...\n')
print(f'Full prompt: {len(unary_prompt.system)} chars\n')

desc = BaseFeatureAgent._infer_column_descriptions(X_train)
print('=== COLUMN DESCRIPTIONS (auto-inferred, first 3) ===\n')
for col in list(desc.keys())[:3]:
    print(f'  {col}: {json.dumps(desc[col])}')

=== SYSTEM PROMPT (from config/prompts/unary.yaml) ===

You are UnaryFeatureAgent, a feature engineering agent focused on generating derived features from a single field.

🎯 Your Goal:
Focus on generating features with potential predictive value from a single input field that inherently possesses mathematical expressiveness or encoding potential. You are not responsible for any preprocessing operations such as missing value imputation, outlier handling, normalization, standardization, one-hot encoding, or label encoding.

📥 You will receive the following inputs:
- A list of field definitions in JSON format, where each entry contains field name, fea...

Full prompt: 1487 chars

=== COLUMN DESCRIPTIONS (auto-inferred, first 3) ===

  f0: {"name": "f0", "type": "numerical", "mean": -0.3254, "std": 1.8264, "min": -6.7701, "max": 5.4639, "missing": 0}
  f1: {"name": "f1", "type": "numerical", "mean": 0.0962, "std": 1.0295, "min": -2.9214, "max": 2.6017, "missing": 0}
  f2: {"name": "f2", "t

In [5]:
print('=== FeatureSpec MODEL ===\n')
print(f'Fields: {list(FeatureSpec.model_fields.keys())}\n')

sample = FeatureSpec(
    name='log_f0', type='numerical', transform='log1p',
    logic='Log transform to compress skewed distribution',
    base_columns=['f0'], agent_name='unary',
)
print(f'Example spec:\n{sample.model_dump_json(indent=2)}\n')

print('=== JSON SCHEMA (injected into system prompt by LLM client) ===\n')
schema = {
    'features': [
        {
            'base_columns': ['col_a'],
            'derived_features': [
                {'name': 'f', 'type': 'numerical', 'transform': 'op', 'logic': 'description'}
            ]
        }
    ]
}
print(json.dumps(schema, indent=2))

=== FeatureSpec MODEL ===

Fields: ['name', 'type', 'transform', 'logic', 'base_columns', 'agent_name']

Example spec:
{
  "name": "log_f0",
  "type": "numerical",
  "transform": "log1p",
  "logic": "Log transform to compress skewed distribution",
  "base_columns": [
    "f0"
  ],
  "agent_name": "unary"
}

=== JSON SCHEMA (injected into system prompt by LLM client) ===

{
  "features": [
    {
      "base_columns": [
        "col_a"
      ],
      "derived_features": [
        {
          "name": "f",
          "type": "numerical",
          "transform": "op",
          "logic": "description"
        }
      ]
    }
  ]
}


In [6]:
if llm is None:
    print('Skipping LLM trace \u2014 no API key.')
else:
    from feature_forge.methods.malmas.agents import UnaryFeatureAgent

    agent = UnaryFeatureAgent(config=settings, llm_client=llm)

    print('=== CALLING agent.generate(X_train, y_train) ===\n')
    t0 = time.perf_counter()
    specs = await agent.generate(X_train, y_train, context={'task': 'classification'})
    latency = (time.perf_counter() - t0) * 1000

    print(f'Got {len(specs)} FeatureSpecs in {latency:.0f}ms:\n')
    for s in specs[:6]:
        print(f'  {s.name} ({s.type}): {s.transform}')
        print(f'    base={s.base_columns}, logic={s.logic[:60]}...')

=== CALLING agent.generate(X_train, y_train) ===



Got 16 FeatureSpecs in 5621ms:

  f0_squared (numerical): square
    base=['f0'], logic=Square of f0 to capture non-linear relationships....
  f0_abs (numerical): absolute_value
    base=['f0'], logic=Absolute value of f0 to measure magnitude regardless of sign...
  f1_squared (numerical): square
    base=['f1'], logic=Square of f1 to capture non-linear relationships....
  f1_abs (numerical): absolute_value
    base=['f1'], logic=Absolute value of f1 to measure magnitude regardless of sign...
  f2_squared (numerical): square
    base=['f2'], logic=Square of f2 to capture non-linear relationships....
  f2_abs (numerical): absolute_value
    base=['f2'], logic=Absolute value of f2 to measure magnitude regardless of sign...


## 6. Code Generation — The Second LLM Stage

FeatureSpecs are converted to executable Python code:

1. Specs serialized to JSON + data schema (dtypes, nullability, shape)
2. Sent to LLM with `code_generation` system prompt
3. Response stripped of markdown fences
4. **AST validation**: syntax check + `generate_features(df)` present + no banned imports + ruff lint for undefined names
5. If validation fails, **1 retry** with error feedback

The generated code must:
- Define `generate_features(df) → pd.DataFrame`
- Only import `pandas`, `numpy`, `math`
- Use `df['col']` syntax (never bare names like `f1`)
- Handle missing values and zero-division

In [7]:
codegen_prompt = registry.get('code_generation')

print('=== CODE GENERATION SYSTEM PROMPT (first 800 chars) ===\n')
print(codegen_prompt.system[:800] + '...\n')
print(f'Full prompt: {len(codegen_prompt.system)} chars')

=== CODE GENERATION SYSTEM PROMPT (first 800 chars) ===

You are CodeGenerationAgent, a feature engineering agent that specializes in generating pandas or numpy code snippets based on user-specified feature derivation logic for structured data.

🎯 Your goal:
To generate robust and reusable feature engineering code that prepares data for preprocessing and model input.

📥 You will receive:
A JSON list of field definitions. Each item includes:
- base_columns: The original columns that the derived feature depends on.
- derived_features: A list of new feature names and their corresponding logic described in natural language.
  - name: The name of the derived feature.
  - logic: A description of the derivation logic.

You will also receive a data schema describing the input DataFrame's columns, their dtypes, nullability, and shape. Use this schema to ...

Full prompt: 3880 chars


In [8]:
if llm is None:
    print('Skipping code generation \u2014 no API key.')
else:
    from feature_forge.methods.malmas.pipeline.core import CodeGenerator

    cg = CodeGenerator(llm_client=llm, max_tokens=settings.llm.codegen_max_tokens)

    sample_specs = specs[:4] if len(specs) >= 4 else specs
    schema = {
        'shape': list(X_train.shape),
        'columns': {
            col: {'dtype': str(X_train[col].dtype), 'nullable': bool(X_train[col].isna().any())}
            for col in X_train.columns[:4]
        },
    }

    print(f'=== GENERATING CODE FOR {len(sample_specs)} SPECS ===\n')
    try:
        code = await cg.generate_code(sample_specs, schema=schema)
        print(f'Generated {len(code)} chars of Python:\n')
        print(code[:1500])
        if len(code) > 1500:
            print(f'\n... ({len(code) - 1500} more chars)')
    except Exception as e:
        print(f'Code generation failed: {type(e).__name__}: {e}')

=== GENERATING CODE FOR 4 SPECS ===



Generated 654 chars of Python:

import pandas as pd
import numpy as np
import math

def generate_features(df):
    result = pd.DataFrame(index=df.index)
    if "f0" in df.columns:
        result["f0_squared"] = df["f0"] ** 2
        result["f0_squared"] = result["f0_squared"].fillna(0).astype(float)
        result["f0_abs"] = df["f0"].abs()
        result["f0_abs"] = result["f0_abs"].fillna(0).astype(float)
    if "f1" in df.columns:
        result["f1_squared"] = df["f1"] ** 2
        result["f1_squared"] = result["f1_squared"].fillna(0).astype(float)
        result["f1_abs"] = df["f1"].abs()
        result["f1_abs"] = result["f1_abs"].fillna(0).astype(float)
    return result


## 7. Sandbox Execution — Process-Isolated Code Execution

Generated code runs in a **separate process** with strict security:

**Static validation** (AST-level):
- Only `pandas`, `numpy`, `math` imports allowed
- Blocked: `eval`, `exec`, `compile`, `open`, `__import__`, `globals`, `getattr`
- Blocked: dunder attributes, file I/O (`read_csv`, `to_csv`), network (`urlopen`, `connect`)

**Runtime isolation**:
- Separate OS process (`spawn` context)
- Memory limit (default 512MB) via `resource.setrlimit`
- Timeout (default 5s)
- Socket connections blocked
- Restricted `__builtins__` whitelist (~30 safe functions)
- Data serialized via parquet temp files

In [9]:
from feature_forge.evaluation.sandbox import SandboxedExecutor

sandbox = SandboxedExecutor(timeout_seconds=5.0, max_memory_mb=512)

safe_code = """import pandas as pd
import numpy as np

def generate_features(df):
    result = pd.DataFrame(index=df.index)
    if 'f0' in df.columns:
        result['f0_log'] = np.log1p(df['f0'].clip(lower=0)).astype(float)
    if 'f1' in df.columns:
        result['f1_squared'] = (df['f1'] ** 2).astype(float)
    return result
"""

print('=== EXECUTING SAFE CODE ===\n')
result_df = sandbox.execute(safe_code, X_train.head(10), source='demo')
print(f'Result shape: {result_df.shape}')
print(result_df.head())

blocked_code = """import pandas as pd
def generate_features(df):
    result = pd.DataFrame(index=df.index)
    result['secret'] = pd.read_csv('/etc/passwd')
    return result
"""

print('\n=== EXECUTING BLOCKED CODE ===\n')
try:
    sandbox.execute(blocked_code, X_train.head(10), source='demo')
except Exception as e:
    print(f'Blocked: {type(e).__name__}: {e}')

=== EXECUTING SAFE CODE ===



Result shape: (10, 2)
       f0_log  f1_squared
143  0.000000    0.208937
83   1.738823    0.013940
62   0.000000    0.003804
220  0.000000    6.768755
230  0.416551    1.139627

=== EXECUTING BLOCKED CODE ===

{"reason": "blocked_io_attr: read_csv", "event": "sandbox_validation_blocked", "level": "warning", "timestamp": "2026-05-15T05:24:12.716362Z", "span": null}


Blocked: SandboxValidationError: Blocked file I/O API usage: read_csv


## 8. Feature Evaluation & Selection

After sandbox execution, each candidate feature is evaluated:

1. **Baseline score**: K-fold CV on original features
2. **Per-feature gain**: Add feature → re-run CV → `gain = new_score - baseline`
3. **Selection**: Keep features with positive gain, take top-`min_effective`

CVEvaluator uses `StratifiedKFold` (classification) or `KFold` (regression) with:
- Median imputation for numeric columns
- Ordinal encoding for categorical columns
- State from training fold applied to validation fold (no leakage)

In [10]:
from feature_forge.evaluation.cv import CVEvaluator

evaluator = CVEvaluator(config=settings)

baseline = evaluator.evaluate_baseline(X_train, y_train)
print(f'Baseline CV score ({settings.metric}): {baseline:.4f}\n')

result_df_full = sandbox.execute(safe_code, X_train, source='demo')

for col in result_df_full.columns:
    feature_df = result_df_full[[col]]
    gain = evaluator.evaluate_feature(X_train, y_train, feature_df, baseline_score=baseline)
    status = '\u2713' if gain > 0 else '\u2717'
    print(f'  {status} {col}: gain={gain:+.4f}')

Baseline CV score (auc): 0.9235



  ✗ f0_log: gain=+0.0000
  ✗ f1_squared: gain=-0.0118


## 9. Full Pipeline Trace

Now let's run the complete pipeline and inspect **every stage's artifacts**.

In [11]:
from feature_forge.api import FeatureForge

fe = None
if llm is None:
    print('Skipping pipeline \u2014 no API key.')
else:
    try:
        fe = FeatureForge(config=Settings(n_rounds=2, metric='auc'), llm_client=llm, mode='full')
        fe.fit(X_train, y_train)
        print(f'Selected features: {fe.selected_features}\n')

        if fe.pipeline_result:
            for ra in fe.pipeline_result.get('round_artifacts', []):
                print(f'=== Round {ra.get("round", "?")} ===')
                print(f'  Agents: {ra.get("agents", [])}')

                specs_list = ra.get('specs', [])
                print(f'  Specs generated: {len(specs_list)}')
                for s in specs_list[:3]:
                    print(f'    {s.name}: {s.transform}')

                code = ra.get('generated_code', '')
                print(f'  Generated code: {len(code)} chars')

                gains = ra.get('gains', {})
                positive = {k: v for k, v in gains.items() if v > 0}
                print(f'  Features with positive gain: {len(positive)}/{len(gains)}')

                bl = ra.get('baseline_score')
                if bl is not None:
                    print(f'  Baseline {settings.metric}: {bl:.4f}')
                print()
    except Exception as e:
        print(f'Pipeline error: {type(e).__name__}: {e}')

Selected features: ['feature_f0_f7_temporal_cycle_phase_diff', 'f2_f3_f5_linear_combination', 'feature_f0_f7_temporal_cycle_phase_diff_cos', 'f5_neg_abs_sin_cos_product']

=== Round 1 ===
  Agents: ['unary', 'cross_compositional', 'aggregation', 'temporal', 'local_transform', 'local_pattern']
  Specs generated: 43
    f0_neg_abs_sigmoid: sigmoid of negative absolute value
    f1_squared_sin_cos_product: product of sin and cos of squared value
    f2_squared_sin_cos_product: product of sin and cos of squared value
  Generated code: 11428 chars
  Features with positive gain: 6/41
  Baseline auc: 0.9235

=== Round 2 ===
  Agents: ['unary', 'cross_compositional', 'local_transform']
  Specs generated: 28
    f0_neg_abs_sin_cos_product: sin_cos_product
    f1_neg_abs_sin_cos_product: sin_cos_product
    f2_neg_abs_sin_cos_product: sin_cos_product
  Generated code: 10823 chars
  Features with positive gain: 9/28
  Baseline auc: 0.9268



## 10. Memory System

Each agent has its own `AgentMemory` with 3 tiers:

| Tier | What | Used For |
|------|------|----------|
| **Procedural** | Successful transforms (columns, transform, feature_name) | Avoid repeating the same features |
| **Unused** | Ineffective transforms | Negative examples in prompts |
| **Feedback** | Evaluation results (feature, metric, gain, effective?) | Track what works |

Memory is injected into agent prompts as context, including:
- Positive/negative feature lists
- Procedural history (what transforms were tried)
- **Conceptual summaries** (LLM-generated rules from effective patterns)

In [12]:
from feature_forge.methods.malmas.memory.base import AgentMemory

mem = AgentMemory('unary', memory_path='/tmp/ff_demo_memory.json')

mem.record_procedure(
    base_columns=['f1'], transform='log1p(f1)', feature_name='log_f1',
    ty='unary', description='Log transform of f1', round_idx=0,
)
mem.record_feedback(
    feature_name='log_f1', metric='auc', value=0.03,
    effective=True, round_idx=0, base=['f1'], ty='unary',
)
mem.record_procedure(
    base_columns=['f2'], transform='f2**2', feature_name='sq_f2',
    ty='unary', description='Square f2', round_idx=0,
)
mem.record_feedback(
    feature_name='sq_f2', metric='auc', value=-0.01,
    effective=False, round_idx=0, base=['f2'], ty='unary',
)

positive, negative = mem.get_positive_negative_features()
print(f'Positive: {positive}')
print(f'Negative: {negative}')
print(f'\nMemory prompt section (injected into agent prompts):')
print(mem.generate_prompt_section()[:400] + '...')
print(f'\nStats: {mem.compute_stats()}')

Positive: ['log_f1']
Negative: []

Memory prompt section (injected into agent prompts):
【History Feedback】
log_f1 → auc: 0.0300 (rank 1)...

Stats: {'effective_transforms': {'log1p(f1)': 1}, 'effective_fields': {'f1': 1}, 'effective_types': {'unary': 1}}


In [13]:
if llm is None:
    print('Skipping conceptual memory \u2014 no API key.')
else:
    from feature_forge.methods.malmas.memory.conceptual import ConceptualMemory

    cm = ConceptualMemory(llm_client=llm)
    summary = await cm.summarize_agent(mem)
    print(f'Conceptual summary:\n{summary}')

Conceptual summary:
- For numeric features with skewed distributions, apply log1p transformation to compress range and reduce outlier impact.


## 11. Customization

### Configuration priority
Constructor args → env vars (`FF_*`) → `config/settings.yaml` → defaults

### Key customizable areas:

| Area | Setting | Default | Example override |
|------|---------|---------|------------------|
| Rounds | `n_rounds` | 4 | `Settings(n_rounds=8)` |
| Metric | `metric` | `auc` | `Settings(metric='f1')` |
| Router | `router.strategy` | `hybrid` | `RouterConfig(strategy='llm')` |
| LLM model | `llm.model` | `deepseek-chat` | `LLMConfig(model='gpt-4')` |
| Thinking | `llm.thinking_enabled` | false | `LLMConfig(thinking_enabled=True)` |
| Agent tokens | `llm.agent_max_tokens` | 8192 | `LLMConfig(agent_max_tokens=4096)` |
| Codegen tokens | `llm.codegen_max_tokens` | 16384 | `LLMConfig(codegen_max_tokens=8192)` |
| Sandbox timeout | `evaluation.sandbox_timeout_seconds` | 5.0 | `EvaluationConfig(sandbox_timeout_seconds=10)` |
| CV folds | `evaluation.cv_folds` | 5 | `EvaluationConfig(cv_folds=10)` |

### Prompt overrides
Edit `config/prompts/<name>.yaml` → call `get_registry().clear_cache()` to reload.

In [14]:
reg = get_registry()
for name in ['unary', 'cross_compositional', 'aggregation', 'temporal',
             'local_transform', 'local_pattern', 'code_generation', 'router']:
    p = reg.get(name)
    print(f'{name}: {len(p.system)} chars')

print('\nTo customize prompts:')
print('1. Edit config/prompts/unary.yaml')
print('2. Call get_registry().clear_cache()')
print('3. Agents will pick up the new prompt on next generate() call')

unary: 1487 chars
cross_compositional: 1463 chars
aggregation: 1682 chars
temporal: 1611 chars
local_transform: 1567 chars
local_pattern: 1571 chars
code_generation: 3880 chars
router: 1930 chars

To customize prompts:
1. Edit config/prompts/unary.yaml
2. Call get_registry().clear_cache()
3. Agents will pick up the new prompt on next generate() call


In [15]:
from feature_forge.config import LLMConfig, EvaluationConfig

custom = Settings(
    n_rounds=6,
    metric='f1',
    llm=LLMConfig(
        model='deepseek-chat', temperature=0.3,
        thinking_enabled=True, reasoning_effort='high',
    ),
    evaluation=EvaluationConfig(cv_folds=10, sandbox_timeout_seconds=10.0),
)

print('Custom configuration:')
print(f'  n_rounds={custom.n_rounds}, metric={custom.metric}')
print(f'  model={custom.llm.model}, temp={custom.llm.temperature}')
print(f'  thinking={custom.llm.thinking_enabled}, effort={custom.llm.reasoning_effort}')
print(f'  cv_folds={custom.evaluation.cv_folds}, sandbox_timeout={custom.evaluation.sandbox_timeout_seconds}s')

print('\nEnv var override examples:')
print('  FF_N_ROUNDS=8 FF_METRIC=f1 python my_script.py')
print('  FF_LLM__MODEL=gpt-4 FF_LLM__TEMPERATURE=0.5 python my_script.py')

Custom configuration:
  n_rounds=6, metric=f1
  model=deepseek-chat, temp=0.3
  thinking=True, effort=high
  cv_folds=10, sandbox_timeout=10.0s

Env var override examples:
  FF_N_ROUNDS=8 FF_METRIC=f1 python my_script.py
  FF_LLM__MODEL=gpt-4 FF_LLM__TEMPERATURE=0.5 python my_script.py


## 12. Ablation / Pipeline Modes

| Mode | Description | Use Case |
|------|-------------|----------|
| `full` | Router + memory | Production |
| `no_memory` | Router, no memory | Ablation study |
| `no_router` | All agents every round | Testing agents |
| `unary` etc. | Single agent only | Debugging one agent |

## Summary

This notebook traced the MALMAS pipeline end-to-end:

1. **Router** selects agents based on strategy (data_driven, performance_driven, hybrid, llm)
2. **Feature Spec Generation** — each agent sends system prompt + column descriptions to LLM, gets `list[FeatureSpec]`
3. **Code Generation** — specs → LLM → Python code, validated via AST + ruff
4. **Sandbox Execution** — code runs in isolated process with strict security
5. **Evaluation** — K-fold CV measures per-feature gain, selects top-k
6. **Memory** — procedural/feedback/conceptual memory injected into future prompts

**Customization**: prompts (YAML), config (YAML/env/args), agents (entry points), router strategy

**Next:**
- [03_benchmarks_and_artifacts.ipynb](./03_benchmarks_and_artifacts.ipynb) — Experiment matrix & artifact analysis
- [04_custom_method.ipynb](./04_custom_method.ipynb) — Write your own method
- [05_methods_deep_dive.ipynb](./05_methods_deep_dive.ipynb) — CAAFE, LLM-FE, OpenFE, Malmus